# TrialSieve: LoRA fine-tuning on Kaggle
Settings: Accelerator **GPU T4 x2**, Internet **On**, secret `HF_TOKEN` attached. Run cells top to bottom.

In [ ]:
!rm -rf trialsieve && git clone -q https://github.com/ibrahimmlk/trialsieve.git
%cd trialsieve
!git log --oneline -1
!pip install -q -U peft
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 makes peft refuse to load; it is unused here
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # single T4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))

## 1. Base model on validation (before fine-tuning)

In [ ]:
!python scripts/evaluate.py --split val --tag base

## 2. LoRA fine-tuning (train split only, validation loss for checkpoint selection)

In [ ]:
!python scripts/train.py --output-dir outputs/lora-qwen1.5b

## 3. Fine-tuned model on validation

In [ ]:
!python scripts/evaluate.py --split val --adapter outputs/lora-qwen1.5b/final --tag finetuned

## 4. Upload adapter and validation results to Hugging Face

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); repo = 'ibrahimmlk/trialsieve-qwen2.5-1.5b-lora'
api.create_repo(repo, exist_ok=True)
api.upload_folder(repo_id=repo, folder_path='outputs/lora-qwen1.5b/final')
for f in ['log_history.json', 'train_config.json']:
    api.upload_file(repo_id=repo, path_or_fileobj=f'outputs/lora-qwen1.5b/{f}', path_in_repo=f'training/{f}')
api.upload_folder(repo_id=repo, folder_path='results', path_in_repo='results')
print('https://huggingface.co/' + repo)

## 5. FINAL held-out test evaluation
Run **only** once the model and prompt are frozen from validation results. Never tune after this.

In [ ]:
!python scripts/evaluate.py --split test --final --tag base
!python scripts/evaluate.py --split test --final --adapter outputs/lora-qwen1.5b/final --tag finetuned
api.upload_folder(repo_id=repo, folder_path='results', path_in_repo='results')